In [96]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import os

In [97]:
data_path = "/home/phile/development/datasets/GuavaDiseaseDataset"
os.listdir(data_path)

['val', 'test', 'train']

In [98]:
labels = os.listdir(data_path)

In [99]:
labels[0]

'val'

In [100]:
transform = transforms.Compose([
    transforms.Resize((224,224,)),
    transforms.ToTensor(), # THIS IS WHERE DATA AUGMENTATION WOULD OCCUR
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=os.path.join(data_path, labels[2]), transform = transform)
val_dataset = ImageFolder(root=os.path.join(data_path, labels[0]), transform = transform)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [101]:
model = models.resnet18(pretrained=True)

num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [102]:
device

'cuda'

In [103]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [104]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        
        #validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                #accuracy
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch +1} / {num_epochs},"
              f"Train loss: {running_loss/len(train_loader):.4f},"
              f"Val loss: {val_loss/len(val_loader):.4f},"
              f"Accuracy: {100 * correct / total:.2f}%"
              )



In [105]:
train_model(model, train_dataloader, val_dataloader, criterion, optimizer)

Epoch 1 / 10,Train loss: 0.1632,Val loss: 0.5138,Accuracy: 83.58%
Epoch 2 / 10,Train loss: 0.0573,Val loss: 0.0839,Accuracy: 97.09%
Epoch 3 / 10,Train loss: 0.0483,Val loss: 0.2517,Accuracy: 93.77%
Epoch 4 / 10,Train loss: 0.0513,Val loss: 1.7083,Accuracy: 74.30%
Epoch 5 / 10,Train loss: 0.0327,Val loss: 0.0503,Accuracy: 97.88%
Epoch 6 / 10,Train loss: 0.0336,Val loss: 0.0514,Accuracy: 98.01%
Epoch 7 / 10,Train loss: 0.0277,Val loss: 0.0505,Accuracy: 98.68%
Epoch 8 / 10,Train loss: 0.0252,Val loss: 0.0666,Accuracy: 97.75%
Epoch 9 / 10,Train loss: 0.0556,Val loss: 0.9796,Accuracy: 78.68%
Epoch 10 / 10,Train loss: 0.0425,Val loss: 0.0180,Accuracy: 99.47%


In [106]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [107]:
from PIL import Image

def predict_image(image_path, model):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)
    return train_dataset.classes[predicted.item()]


In [108]:
my_img = "/home/phile/development/datasets/GuavaDiseaseDataset/test/fruit_fly/20230622_153013_unsharp_clahe_augmented_7.png"
print(predict_image(my_img, model))

fruit_fly


In [109]:
my_img = "/home/phile/development/datasets/GuavaDiseaseDataset/test/healthy_guava/47_unsharp_clahe_augmented_3.png"
print(predict_image(my_img, model))

healthy_guava
